In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('Квартиры_clean_v5.csv')

print(f"Всего квартир: {len(df)}")
print(f"Уникальных домов: {df['cluster'].nunique()}")
print(f"Районов: {df['Район'].nunique()}")
print(f"Районы: {df['Район'].unique()}")
print("="*50)

all_houses = df.groupby('Район')['cluster'].nunique().reset_index()
all_houses.columns = ['Район', 'Всего_домов']

all_flats = df.groupby('Район').size().reset_index(name='Всего_квартир')

novostroyki = df[df['Тип.объекта'] == 'Новостройка']
new_houses = novostroyki.groupby('Район')['cluster'].nunique().reset_index()
new_houses.columns = ['Район', 'Новостроек_домов']

new_flats = novostroyki.groupby('Район').size().reset_index(name='Новостроек_квартир')

median_price = df.groupby('Район')['Цена.за.м'].median().reset_index()
median_price.columns = ['Район', 'Медиана_цена_м2']

median_price_new = novostroyki.groupby('Район')['Цена.за.м'].median().reset_index()
median_price_new.columns = ['Район', 'Медиана_цена_м2_нов']

vtorichka = df[df['Тип.объекта'] == 'Вторичная']
median_price_sec = vtorichka.groupby('Район')['Цена.за.м'].median().reset_index()
median_price_sec.columns = ['Район', 'Медиана_цена_м2_втор']

avg_year = df.groupby('Район')['Год.постройки'].mean().reset_index()
avg_year.columns = ['Район', 'Средний_год_постройки']

rating = all_houses.copy()
for table in [all_flats, new_houses, new_flats, median_price,
              median_price_new, median_price_sec, avg_year]:
    rating = rating.merge(table, on='Район', how='left')

rating['Новостроек_домов'] = rating['Новостроек_домов'].fillna(0).astype(int)
rating['Новостроек_квартир'] = rating['Новостроек_квартир'].fillna(0).astype(int)

rating['Доля_новостроек'] = (rating['Новостроек_домов'] / rating['Всего_домов']).round(3)

rating = rating.sort_values('Медиана_цена_м2', ascending=False)

print("\n===== СВОДНАЯ ТАБЛИЦА ПО РАЙОНАМ =====\n")
print(rating.to_string(index=False))
print("="*50)



In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

metrics = ['Медиана_цена_м2', 'Доля_новостроек', 'Средний_год_постройки']
rating[['norm_цена', 'norm_доля_нов', 'norm_год']] = scaler.fit_transform(
    rating[metrics].fillna(0)
)

w_price = 0.75
w_new = 0.20
w_year = 0.05

rating['Индекс'] = (
    w_price * rating['norm_цена'] +
    w_new * rating['norm_доля_нов'] +
    w_year * rating['norm_год']
).round(3)

rating = rating.sort_values('Индекс', ascending=False)
rating['Ранг'] = range(1, len(rating) + 1)

print("\n===== РЕЙТИНГ РАЙОНОВ =====\n")
print(rating[['Ранг', 'Район', 'Всего_домов', 'Новостроек_домов',
              'Доля_новостроек', 'Медиана_цена_м2', 'Средний_год_постройки',
              'Индекс']].to_string(index=False))

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

ax1 = axes[0, 0]
r = rating.sort_values('Индекс')
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(r)))
ax1.barh(r['Район'], r['Индекс'], color=colors)
ax1.set_title('Индекс привлекательности районов', fontsize=14)
ax1.set_xlabel('Индекс')

ax2 = axes[0, 1]
r2 = rating.sort_values('Медиана_цена_м2')
ax2.barh(r2['Район'], r2['Медиана_цена_м2'], color='#E69F00')
ax2.set_title('Медианная цена за м²', fontsize=14)
ax2.set_xlabel('Руб./м²')

ax3 = axes[1, 0]
r3 = rating.sort_values('Доля_новостроек')
ax3.barh(r3['Район'], r3['Доля_новостроек'] * 100, color='#56B4E9')
ax3.set_title('Доля новостроек (%)', fontsize=14)
ax3.set_xlabel('%')

ax4 = axes[1, 1]
ax4.scatter(rating['Доля_новостроек'] * 100, rating['Медиана_цена_м2'],
            s=rating['Всего_домов'] * 2, color='#009E73', alpha=0.7)
for _, row in rating.iterrows():
    ax4.annotate(row['Район'],
                (row['Доля_новостроек'] * 100, row['Медиана_цена_м2']),
                fontsize=8, ha='left', va='bottom')
ax4.set_xlabel('Доля новостроек (%)')
ax4.set_ylabel('Медианная цена за м²')
ax4.set_title('Цена vs Доля новостроек\n(размер = кол-во домов)', fontsize=14)

plt.tight_layout()
plt.savefig('rating_districts.png', dpi=150, bbox_inches='tight')
plt.show()

from scipy.stats import spearmanr

corr, pval = spearmanr(rating['Индекс'], rating['Медиана_цена_м2'])
print(f"\nКорреляция Спирмена (Индекс vs Цена): {corr:.3f}, p-value: {pval:.4f}")

corr2, pval2 = spearmanr(rating['Доля_новостроек'], rating['Медиана_цена_м2'])
print(f"Корреляция Спирмена (Доля новостроек vs Цена): {corr2:.3f}, p-value: {pval2:.4f}")

rating.to_csv('Рейтинг_районов2.csv', index=False, encoding='utf-8-sig')
print("\nРейтинг сохранён: Рейтинг_районов.csv")
from scipy.stats import pearsonr

district_features = rating[['Район', 'norm_доля_нов', 'norm_год', 'norm_цена']].copy()

df_m = df.merge(district_features, on='Район', how='left')

print(f"Наблюдений: {len(df_m)}")

target = df_m['Цена.за.м']

print("\n===== КОРРЕЛЯЦИЯ ПИРСОНА (уровень квартир) =====\n")
print(f"{'Метрика':<20} {'p-value':>14} {'Значим?':>10}")
print("-" * 55)

for col in ['norm_доля_нов', 'norm_год', 'norm_цена']:
    r, p = pearsonr(df_m[col], target)
    sig = "✅ Да" if p < 0.05 else "❌ Нет"
    print(f"{col:<20} {p:>14.8f} {sig:>10}")